# 06 native yolo26m to yolo26n — Google Colab

1. Set **Runtime > Change runtime type > GPU** for training.
2. Edit `DATASET_ZIP` and `DRIVE_OUTPUT_ROOT` below.
3. Add `WANDB_API_KEY` in **Colab Secrets** and enable notebook access (training only).
4. Run All. Research hyperparameters and seed schedules are in the next configuration cell.

Source: `05_native_yolo26n.ipynb`. Experiment IDs/artifact names are preserved. This copy is standalone; no repository checkout is required.

### Method notes

**Eksperimen 05** · Native feature distillation bawaan Ultralytics (score-weighted L2).

Notebook ini memuat **satu eksperimen**. Dataset awal `coco8.yaml` hanyalah uji integrasi (bukan hasil penelitian). Setelah dataset penelitian siap, ubah `DATA` dan parameter training di section 1. Audit dataset berada **di luar notebook ini**.

**Urutan:** jalankan notebook 01 dahulu; lalu gunakan `weights/best.pt` teacher yang sama untuk eksperimen 04–09. **Adaptasi CrossKD/CSAKD bukan reproduksi identik paper.**

**Demo W&B:** COCO8 (4 gambar train / 4 val), 10 epoch; model checkpoint setiap epoch sebagai W&B Artifact berversi. Angka mAP **bukan** hasil penelitian.


## Colab paths

Keep the ZIP unchanged in Drive. Extraction and the derived YAML are local; source images, labels and YAML are not edited. For multiple YAML files, set `DATASET_YAML_RELATIVE` to the desired archive-relative path.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import uuid

# Edit these two paths to match your Google Drive.
DATASET_ZIP = "/content/drive/MyDrive/Skripsi/datasets/poltrada-v3-yolo26-clean.zip"
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/Skripsi/yolo26-kd-indotraffic/results"
DATASET_YAML_RELATIVE = ""  # Set only if the ZIP contains more than one YAML.
SOURCE_DATA_DECLARATION = '/kaggle/input/datasets/ilhamrafiqin/poltrada-v3-yolo26-clean/data.yaml'  # Original teacher provenance; not an active Colab path.
COLAB_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_" + uuid.uuid4().hex[:6]
COLAB_LOCAL_DIR = Path("/content/yolo26_colab") / COLAB_RUN_ID
COLAB_OUTPUT_DIR = Path(DRIVE_OUTPUT_ROOT) / "06_native_yolo26m_to_yolo26n" / COLAB_RUN_ID
COLAB_DATA = str(COLAB_LOCAL_DIR / "runtime/data.yaml")


## Research configuration

Preserved from the original notebook. The trio defaults to 100 epochs with smoke disabled. Existing multi-seed queues remain unchanged; 100 epochs is per seed.

In [ ]:
# Ubah parameter di bagian ini saja untuk menjalankan eksperimen.
EXPERIMENT_ID = "05_native_yolo26n_dis1"
METHOD = "native"
DATA = COLAB_DATA  # Runtime YAML prepared from the Drive ZIP.
MODEL_WEIGHTS = "yolo26n.pt"
EPOCHS = 100
IMGSZ = 640
BATCH = 16
SEED = 42
DEVICE = 0                   # GPU Colab; 'cpu' untuk diagnosis lokal
WORKERS = 2
OPTIMIZER = "AdamW"
LR = 0.001
PATIENCE = 100
AMP = True

WEIGHT_DECAY = 0.0005
WARMUP_EPOCHS = 3.0
CLOSE_MOSAIC = 10
EVAL_SPLIT = "val"           # test hanya jika dataset punya split test
PROJECT = str(COLAB_OUTPUT_DIR)  # Persistent Google Drive output for this run.
TEACHER_CHECKPOINT = ""   # Diisi otomatis dari artifact W&B v50 di section 4.
KD_WEIGHT = 1.0            # dis: official Ultralytics KD

# W&B online: setiap epoch mengunggah last.pt + best.pt sampai epoch tersebut.
WANDB_ENABLED = True
WANDB_PROJECT = "yolom26-kd-indotraffic"
WANDB_ENTITY = None          # gunakan akun default; ganti nama tim bila perlu

# Group membandingkan pasangan yang sama; tag mengikuti konfigurasi run baru.
WANDB_GROUP = "T26m_S26n"
WANDB_TAGS = [
    METHOD, 'teacher-m', 'student-n', f"dis{KD_WEIGHT:g}", f"seed{SEED}", "poltrada-v3", EXPERIMENT_ID
]


## Colab: Drive, dependencies, and local dataset

Train from the local extracted ZIP; outputs go to Drive and existing W&B callbacks. Authorize Drive mounting when prompted.

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
COLAB_LOCAL_DIR.mkdir(parents=True, exist_ok=False)
COLAB_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(COLAB_LOCAL_DIR)  # Downloads and temporary files stay on the Colab VM.
os.environ["WANDB_CACHE_DIR"] = str(COLAB_LOCAL_DIR / "wandb-cache")
os.environ["WANDB_DIR"] = str(COLAB_LOCAL_DIR)
os.environ["PIP_CACHE_DIR"] = str(COLAB_LOCAL_DIR / "pip-cache")
os.environ["TORCH_HOME"] = str(COLAB_LOCAL_DIR / "torch-cache")
print("Persistent output:", COLAB_OUTPUT_DIR)


In [ ]:
from importlib import metadata
import subprocess
import sys

# Preserve Colab's installed PyTorch/torchvision build during dependency resolution.
constraints = []
for package in ("torch", "torchvision"):
    try:
        constraints.append(package + "==" + metadata.version(package))
    except metadata.PackageNotFoundError:
        if True:
            raise RuntimeError("Use a Colab GPU runtime with PyTorch installed.") from None
constraint_file = COLAB_LOCAL_DIR / "colab-torch-constraints.txt"
constraint_file.write_text("\n".join(constraints), encoding="utf-8")
for distribution, package, version in [('ultralytics', 'ultralytics', '8.4.155'), ('wandb', 'wandb', '0.28.1'), ('PyYAML', 'PyYAML', None)]:
    try:
        installed = metadata.version(distribution)
    except metadata.PackageNotFoundError:
        installed = None
    if installed is None or (version is not None and installed != version):
        requirement = package if version is None else package + "==" + version
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-c", str(constraint_file), requirement])


In [ ]:
"""Portable helpers embedded into each generated Colab notebook (no repo imports)."""

import hashlib
import os
import shutil
import stat
import zipfile
from pathlib import Path, PurePosixPath

import yaml


def colab_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def colab_api_key(name="WANDB_API_KEY"):
    """Prefer an existing environment value; otherwise request a Colab Secret."""
    value = os.environ.get(name, "").strip()
    if not value:
        try:
            from google.colab import userdata

            value = userdata.get(name)
        except Exception:  # noqa: BLE001 -- Normalize UI failures without exposing secret-bearing payloads.
            raise RuntimeError(
                f"Add {name} in Colab Secrets (key icon), enable notebook access, "
                "then rerun this cell. Never paste the key into notebook code."
            ) from None
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f"Colab Secret {name} is empty; update it and rerun this cell.")
    os.environ[name] = value.strip()
    return value.strip()


def colab_require_gpu(torch_module):
    if not torch_module.cuda.is_available():
        raise RuntimeError("Select Runtime > Change runtime type > GPU in Colab, then Run All.")


def colab_extract_zip(archive, destination):
    """Reject unsafe/duplicate members before extracting; ZIP CRCs validate reads."""
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive) as source:
        members = source.infolist()
        targets = []
        seen = set()
        for member in members:
            name = member.filename.replace("\\", "/")
            relative = PurePosixPath(name)
            if member.is_dir() and str(relative) == ".":
                continue  # Common `zip -r archive.zip .` root-directory entry.
            if (relative.is_absolute() or not relative.parts or ".." in relative.parts
                    or any(":" in part for part in relative.parts)
                    or stat.S_ISLNK(member.external_attr >> 16)):
                raise ValueError(f"Unsafe ZIP member: {member.filename!r}")
            target = destination.joinpath(*relative.parts).resolve()
            if not target.is_relative_to(destination) or target in seen:
                raise ValueError(f"Unsafe or duplicate ZIP member: {member.filename!r}")
            seen.add(target)
            targets.append((member, target))
        needed = sum(member.file_size for member in members)
        if needed > shutil.disk_usage(destination.parent).free:
            raise OSError("Insufficient Colab local disk for the extracted dataset.")
        destination.mkdir(parents=True, exist_ok=False)
        for member, target in targets:
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with source.open(member) as incoming, target.open("wb") as outgoing:
                    shutil.copyfileobj(incoming, outgoing)
        return {"files": sum(not member.is_dir() for member in members), "bytes": needed}


def colab_resolve_reference(value, root, extracted, declared_root, source_declaration):
    """Relocate the declared root, keeping directory names and split membership."""
    value = str(value).replace("\\", "/")
    relative = PurePosixPath(value)
    if relative.is_absolute():
        prefixes = [str(declared_root or ""), str(PurePosixPath(source_declaration).parent)]
        for prefix in prefixes:
            prefix_path = PurePosixPath(prefix)
            if prefix and prefix_path.is_absolute() and relative.is_relative_to(prefix_path):
                relative = relative.relative_to(prefix_path)
                break
        else:
            raise ValueError(f"Cannot relocate absolute dataset path: {value}")
    candidate = root.joinpath(*relative.parts).resolve()
    # Same leading ../ compatibility used by the pinned Ultralytics loader.
    if not candidate.exists() and value.startswith("../"):
        candidate = (root / value[3:]).resolve()
    if not candidate.is_relative_to(extracted) or not candidate.exists():
        raise FileNotFoundError(f"Dataset path is missing or outside the extracted ZIP: {value}")
    return candidate


def prepare_colab_dataset(zip_path, local_directory, yaml_relative, source_declaration):
    """Copy a Drive ZIP locally and write a separate, fully resolved runtime YAML."""
    zip_path, local = Path(zip_path), Path(local_directory).resolve()
    if not zip_path.is_file():
        raise FileNotFoundError(f"Dataset ZIP not found: {zip_path}. Edit DATASET_ZIP in Colab configuration.")
    local.mkdir(parents=True, exist_ok=True)
    runtime_yaml = local / "runtime/data.yaml"
    # A completed setup is immutable within this run; do not overwrite/reuse partial extraction.
    if (local / "source").exists() or runtime_yaml.exists():
        raise RuntimeError("Dataset setup already started for this run. Rerun the Colab configuration cell to create a new run directory.")
    archive = local / "dataset.zip"
    if zip_path.stat().st_size > shutil.disk_usage(local).free:
        raise OSError("Insufficient Colab local disk to copy DATASET_ZIP.")
    shutil.copyfile(zip_path, archive)
    archive_sha = colab_sha256(archive)
    if colab_sha256(zip_path) != archive_sha:
        raise ValueError("Dataset ZIP changed or was corrupted during the copy from Drive.")
    extracted = local / "source"
    inventory = colab_extract_zip(archive, extracted)
    if yaml_relative:
        original_yaml = (extracted / yaml_relative).resolve()
        if not original_yaml.is_relative_to(extracted) or not original_yaml.is_file():
            raise FileNotFoundError("DATASET_YAML_RELATIVE must name a YAML file inside the ZIP.")
    else:
        candidates = sorted(p for p in extracted.rglob("*") if p.suffix.lower() in {".yaml", ".yml"})
        if len(candidates) != 1:
            choices = [str(p.relative_to(extracted)) for p in candidates]
            raise ValueError(f"Expected one dataset YAML, found {choices}. Set DATASET_YAML_RELATIVE explicitly.")
        original_yaml = candidates[0]
    class UniqueDatasetLoader(yaml.SafeLoader):
        pass

    def unique_mapping(loader, node):
        mapping = {}
        for key_node, value_node in node.value:
            key = loader.construct_object(key_node)
            if key in mapping:
                raise ValueError(f"Duplicate dataset YAML key: {key!r}; fix the source ZIP explicitly.")
            mapping[key] = loader.construct_object(value_node)
        return mapping

    UniqueDatasetLoader.add_constructor(yaml.resolver.BaseResolver.DEFAULT_MAPPING_TAG, unique_mapping)
    raw = yaml.load(original_yaml.read_text(encoding="utf-8-sig"), Loader=UniqueDatasetLoader)
    if not isinstance(raw, dict) or not raw.get("train") or not raw.get("val"):
        raise ValueError("Dataset YAML must declare train, val and class names.")
    names = raw.get("names")
    if isinstance(names, dict):
        keys = sorted(names, key=lambda key: int(key))
        if [int(key) for key in keys] != list(range(len(keys))):
            raise ValueError("Class IDs must be consecutive starting at zero.")
        ordered_names = [str(names[key]) for key in keys]
    elif isinstance(names, list):
        ordered_names = [str(name) for name in names]
    else:
        raise TypeError("Dataset YAML names must be a list or a class-ID mapping.")
    if not ordered_names or int(raw.get("nc", len(ordered_names))) != len(ordered_names):
        raise ValueError("Dataset nc and class names disagree.")
    declared_root = raw.get("path")
    root = original_yaml.parent
    if declared_root and not PurePosixPath(str(declared_root).replace("\\", "/")).is_absolute():
        root = (root / declared_root).resolve()
    if not root.is_relative_to(extracted) or not root.is_dir():
        raise ValueError("Dataset YAML root must resolve inside the extracted ZIP.")
    relocated = dict(raw)
    relocated["path"] = str(root)
    runtime_yaml.parent.mkdir(parents=True, exist_ok=True)
    split_counts = {}
    # Match Ultralytics 8.4.155 IMG_FORMATS, including its less common formats.
    image_suffixes = {".avif", ".bmp", ".dng", ".heic", ".heif", ".jp2", ".jpeg", ".jpg", ".mpo", ".png", ".tif", ".tiff", ".webp"}
    for split in ("train", "val", "test"):
        if not raw.get(split):
            continue
        values = raw[split] if isinstance(raw[split], list) else [raw[split]]
        outputs, count = [], 0
        for index, value in enumerate(values):
            if not isinstance(value, str):
                raise TypeError(f"Invalid {split} path: {value!r}")
            path = colab_resolve_reference(value, root, extracted, declared_root, source_declaration)
            if path.is_dir():
                count += sum(p.is_file() and p.suffix.lower() in image_suffixes for p in path.rglob("*"))
                outputs.append(str(path))
            elif path.suffix.lower() == ".txt":
                entries = []
                for line in path.read_text(encoding="utf-8-sig").splitlines():
                    if not line.strip():
                        continue
                    image_root = path.parent if line.strip().startswith("./") else root
                    image = colab_resolve_reference(line.strip(), image_root, extracted, declared_root, source_declaration)
                    if not image.is_file() or image.suffix.lower() not in image_suffixes:
                        raise ValueError(f"Split list references a non-image: {line}")
                    entries.append(str(image))
                derived = runtime_yaml.parent / f"{split}-{index}.txt"
                derived.write_text("\n".join(entries) + "\n", encoding="utf-8")
                outputs.append(str(derived))
                count += len(entries)
            else:
                raise ValueError(f"Split must reference an image directory or TXT list: {value}")
        if count == 0:
            raise ValueError(f"No images found in {split}; check the ZIP and YAML paths.")
        relocated[split] = outputs if isinstance(raw[split], list) else outputs[0]
        split_counts[split] = count
    runtime_yaml.write_text(yaml.safe_dump(relocated, sort_keys=False, allow_unicode=True), encoding="utf-8")
    provenance = {
        "runtime": "google-colab", "drive_zip": str(zip_path), "zip_sha256": archive_sha,
        "extracted_inventory": inventory, "original_yaml": str(original_yaml),
        "original_yaml_sha256": colab_sha256(original_yaml), "runtime_yaml": str(runtime_yaml),
        "runtime_yaml_sha256": colab_sha256(runtime_yaml), "source_dataset_declaration": source_declaration,
        "class_names": ordered_names, "split_image_counts": split_counts,
        "relocation_only": True,
        "identity_note": "ZIP provenance is recorded; equality with the original Kaggle image/label bytes is not independently established.",
    }
    return str(runtime_yaml), provenance


In [ ]:
import json
import torch
colab_require_gpu(torch)
COLAB_DATA, COLAB_DATA_PROVENANCE = prepare_colab_dataset(
    DATASET_ZIP, COLAB_LOCAL_DIR, DATASET_YAML_RELATIVE, SOURCE_DATA_DECLARATION)
DATA = COLAB_DATA
(COLAB_OUTPUT_DIR / "dataset_provenance.json").write_text(
    json.dumps(COLAB_DATA_PROVENANCE, indent=2, ensure_ascii=False), encoding="utf-8")
print("Dataset:", COLAB_DATA)
print("Split image counts:", COLAB_DATA_PROVENANCE["split_image_counts"])


## 2. Environment Setup

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys
import os

# Pertahankan instalasi PyTorch/CUDA bawaan Kaggle; instal hanya yang dibutuhkan.
for package, version in (("ultralytics", "8.4.155"), ("wandb", "0.28.1")):
    try:
        installed = metadata.version(package)
    except metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        print(f"Installing {package}=={version} (was {installed})")
        subprocess.check_call([sys.executable, "-m", "pip", "install", f"{package}=={version}"])

import json
import time
from datetime import datetime
from pathlib import Path

import torch
import wandb
from ultralytics import YOLO, settings
from ultralytics.utils.torch_utils import init_seeds

# Satu sumber logging W&B: callback eksplisit di section 6, bukan integrasi otomatis Ultralytics.
settings.update({"wandb": False})
init_seeds(SEED, deterministic=True)
print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__} | Ultralytics {metadata.version('ultralytics')} | W&B {metadata.version('wandb')}")
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "not available")
if DEVICE != "cpu" and not torch.cuda.is_available():
    raise RuntimeError("GPU tidak tersedia. Aktifkan GPU Colab atau set DEVICE='cpu' untuk test lokal.")

if WANDB_ENABLED:
    os.environ["WANDB_API_KEY"] = colab_api_key()  # Colab atau lokal: sediakan WANDB_API_KEY sebagai environment variable.
    if not os.environ.get("WANDB_API_KEY"):
        raise RuntimeError("W&B online membutuhkan WANDB_API_KEY. Set sebagai Colab Secret atau environment variable; jangan tulis token ke notebook.")
    wandb.login(key=os.environ["WANDB_API_KEY"])


## 3. Dataset Loading

In [ ]:
from ultralytics.data.utils import check_det_dataset

# Memakai loader resmi, tanpa audit/cleaning dataset dalam notebook eksperimen.
data_info = check_det_dataset(DATA)
assert int(data_info["nc"]) > 0, "Dataset tidak memiliki kelas deteksi."
assert data_info.get("train") and data_info.get("val"), "Dataset wajib punya train dan val."
if EVAL_SPLIT == "test" and not data_info.get("test"):
    raise ValueError("Dataset ini tidak memiliki test split; gunakan EVAL_SPLIT='val' hanya untuk diagnosis.")

# Periksa kesesuaian nama kelas teacher/dataset untuk metode KD.
def class_names(names):
    if isinstance(names, dict):
        return [str(names[k]) for k in sorted(names, key=lambda k: int(k))]
    return [str(x) for x in names]

print(f"Dataset: {DATA} | classes: {data_info['nc']} | validation: {EVAL_SPLIT}")


## 4. Model Initialization

In [ ]:
import wandb
from pathlib import Path

api = wandb.Api()

artifact = api.artifact(
    "ilhamrafiqin-brawijaya-university/"
    "yolom26-kd-indotraffic/"
    "01_teacher_yolo26m-fux5wdcb-checkpoints:v50",
    type="model"
)

artifact_dir = Path(
    artifact.download(
        root=str(COLAB_LOCAL_DIR / "teacher")
    )
)

TEACHER_CHECKPOINT = str(artifact_dir / "best.pt")

assert Path(TEACHER_CHECKPOINT).is_file()

print("Teacher:", TEACHER_CHECKPOINT)
print("Artifact:", artifact.name)
print("Epoch:", artifact.metadata.get("epoch"))

if not TEACHER_CHECKPOINT:
    raise ValueError("Isi TEACHER_CHECKPOINT dengan best.pt dari notebook 01 yang dilatih pada dataset yang sama.")
teacher_path = Path(TEACHER_CHECKPOINT).expanduser()
if not teacher_path.is_file():
    raise FileNotFoundError(f"Checkpoint teacher tidak ditemukan: {teacher_path}")
# Model teacher digunakan sebagai target; tidak dilatih ulang oleh notebook student.
teacher_meta = YOLO(str(teacher_path))
if int(teacher_meta.model.nc) != int(data_info["nc"]):
    raise ValueError("Jumlah kelas teacher dan dataset tidak sama.")
if class_names(teacher_meta.model.names) != class_names(data_info["names"]):
    raise ValueError("Urutan nama kelas teacher dan dataset berbeda.")
print("Teacher:", teacher_path, "| classes:", teacher_meta.model.nc)
del teacher_meta

model = YOLO(MODEL_WEIGHTS)
print("Student / model:", MODEL_WEIGHTS)


## 5. Knowledge Distillation Method

Native KD melalui API `distill_model` dan `dis` pada Ultralytics **8.4.155**. Tidak ada custom loss.

In [ ]:
KD_TRAINER = None
KD_TRAIN_ARGS = {"distill_model": str(teacher_path), "dis": KD_WEIGHT}
KD_CONFIG = {"dis": KD_WEIGHT}


## 6. Model Training

In [ ]:
from datetime import datetime

RUN_NAME = f"{EXPERIMENT_ID}_seed{SEED}_{datetime.now():%Y%m%d_%H%M%S}"
train_args = dict(
    data=DATA, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
    optimizer=OPTIMIZER, lr0=LR, seed=SEED, device=DEVICE,
    weight_decay=WEIGHT_DECAY, warmup_epochs=WARMUP_EPOCHS, close_mosaic=CLOSE_MOSAIC,
    project=PROJECT, name=RUN_NAME, exist_ok=False,
    workers=WORKERS, patience=PATIENCE, plots=True,
    amp=AMP, pretrained=True, save=True,
)
train_args.update(KD_TRAIN_ARGS)

# W&B: satu run dan satu koleksi artifact checkpoint berversi per eksperimen.
wb_run = None
wb_state = {"uploaded_epochs": set(), "logged_epochs": set(), "best_epoch": None}
if WANDB_ENABLED:
    wb_run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=RUN_NAME,
        job_type="training",
        group=f"diagnostic-{WANDB_GROUP}" if DATA == "coco8.yaml" else WANDB_GROUP,
        config={**({
            "experiment": EXPERIMENT_ID, "method": METHOD, "model": MODEL_WEIGHTS,
            "teacher_checkpoint": str(TEACHER_CHECKPOINT) if TEACHER_CHECKPOINT else None,
            "dataset": DATA, "eval_split": EVAL_SPLIT, "epochs": EPOCHS,
            "imgsz": IMGSZ, "batch": BATCH, "seed": SEED, "device": str(DEVICE),
            "optimizer": OPTIMIZER, "learning_rate": LR, "amp": AMP,
            "workers": WORKERS, "patience": PATIENCE,
            "weight_decay": WEIGHT_DECAY, "warmup_epochs": WARMUP_EPOCHS,
            "close_mosaic": CLOSE_MOSAIC,
            "kd": KD_CONFIG, "diagnostic_only": DATA == "coco8.yaml",
        }), "colab_dataset": COLAB_DATA_PROVENANCE, "runtime": "google-colab"},
        tags=[*([*WANDB_TAGS, "diagnostic" if DATA == "coco8.yaml" else "research"]), "colab"],
    )

    def save_epoch_artifact(trainer):
        """Snapshot checkpoint setiap epoch; upload selesai sebelum last.pt ditimpa epoch berikutnya."""
        epoch = int(trainer.epoch) + 1
        last_path = Path(trainer.last)
        best_path = Path(trainer.best)
        if not last_path.is_file():
            raise RuntimeError(f"Checkpoint epoch {epoch} tidak ada: {last_path}")
        fitness = None if trainer.fitness is None else float(trainer.fitness)
        best_fitness = None if trainer.best_fitness is None else float(trainer.best_fitness)
        is_best = fitness is not None and best_fitness is not None and fitness == best_fitness
        if is_best:
            wb_state["best_epoch"] = epoch
        artifact = wandb.Artifact(
            name=f"{EXPERIMENT_ID}-{wb_run.id}-checkpoints",
            type="model",
            description="Checkpoint penuh per epoch: last.pt, best-to-date.pt, results.csv, args.yaml",
            metadata={
                "epoch": epoch, "fitness": fitness, "best_fitness": best_fitness,
                "is_best_epoch": bool(is_best), "best_epoch_so_far": wb_state["best_epoch"],
                "experiment": EXPERIMENT_ID, "dataset": DATA, "seed": SEED,
            },
        )
        artifact.add_file(str(last_path), name="last.pt")
        if best_path.is_file():
            artifact.add_file(str(best_path), name="best.pt")
        for filename in ("results.csv", "args.yaml"):
            file_path = Path(trainer.save_dir) / filename
            if file_path.is_file():
                artifact.add_file(str(file_path), name=filename)
        aliases = [f"epoch-{epoch:03d}", "latest"]
        if is_best:
            aliases.append("best")
        # Synchronous wait menjamin isi artifact epoch ini selesai dikirim sebelum file lokal berubah.
        wb_run.log_artifact(artifact, aliases=aliases).wait()
        wb_state["uploaded_epochs"].add(epoch)
        print(f"W&B checkpoint: epoch={epoch}, best={is_best}, aliases={aliases}")

    def log_epoch_metrics(trainer):
        """Satu record grafik per epoch (training loss, KD loss, validation, LR, best epoch)."""
        epoch = int(trainer.epoch) + 1
        # Ultralytics memanggil on_fit_epoch_end sekali lagi pada final_eval; abaikan panggilan ekstra.
        if epoch not in wb_state["uploaded_epochs"] or epoch in wb_state["logged_epochs"]:
            return
        loss = trainer.label_loss_items(trainer.tloss, prefix="train")
        payload = {"epoch": epoch, **{k: float(v) for k, v in loss.items()},
                   **{k: float(v) for k, v in trainer.lr.items()}}
        for key, value in trainer.metrics.items():
            try:
                payload[key] = float(value)
            except (ValueError, TypeError):
                continue
        payload["checkpoint/is_best_epoch"] = int(wb_state["best_epoch"] == epoch)
        payload["checkpoint/best_epoch_so_far"] = wb_state["best_epoch"] or 0
        wb_run.log(payload, step=epoch)
        wb_state["logged_epochs"].add(epoch)

    model.add_callback("on_model_save", save_epoch_artifact)
    model.add_callback("on_fit_epoch_end", log_epoch_metrics)

print("Training:", EXPERIMENT_ID, "| data:", DATA, "| epochs:", EPOCHS,
      "| W&B:", "online + every epoch artifact" if WANDB_ENABLED else "disabled")
start_time = time.perf_counter()
try:
    if KD_TRAINER is None:
        model.train(**train_args)
    else:
        model.train(trainer=KD_TRAINER, **train_args)
except BaseException:
    if wb_run is not None:
        wb_run.finish(exit_code=1)
    raise
training_seconds = time.perf_counter() - start_time
run_dir = Path(model.trainer.save_dir)
best_pt = run_dir / "weights" / "best.pt"
if not best_pt.is_file():
    raise FileNotFoundError(f"Checkpoint best.pt tidak ditemukan: {best_pt}")
if wb_run is not None and len(wb_state["uploaded_epochs"]) != EPOCHS:
    print("Catatan: jumlah epoch selesai berbeda dari target (misalnya early stopping).",
          len(wb_state["uploaded_epochs"]), "/", EPOCHS)
print("Training selesai:", run_dir)


## 7. Model Evaluation

In [ ]:
# Evaluasi dari checkpoint terbaik (bukan last.pt).
best_model = YOLO(str(best_pt))
metrics = best_model.val(
    data=DATA, split=EVAL_SPLIT, imgsz=IMGSZ, batch=BATCH,
    device=DEVICE, plots=True, verbose=False,
)

from ultralytics.utils.torch_utils import get_flops
parameters = sum(p.numel() for p in best_model.model.parameters())
try:
    gflops = float(get_flops(best_model.model, imgsz=IMGSZ))
except (AttributeError, ImportError, RuntimeError, TypeError, ValueError) as error:
    print("GFLOPs tidak tersedia:", error)
    gflops = None

results = {
    "experiment": EXPERIMENT_ID,
    "method": METHOD,
    "model": MODEL_WEIGHTS,
    "dataset": DATA,
    "eval_split": EVAL_SPLIT,
    "seed": SEED,
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "optimizer": OPTIMIZER,
    "learning_rate": LR,
    "precision": float(metrics.box.mp),
    "recall": float(metrics.box.mr),
    "map50": float(metrics.box.map50),
    "map50_95": float(metrics.box.map),
    "parameters": int(parameters),
    "gflops": gflops,
    "inference_ms_per_image": float(metrics.speed.get("inference", 0.0)),
    "training_seconds": float(training_seconds),
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "teacher_checkpoint": str(TEACHER_CHECKPOINT) if METHOD != "none" else None,
    "best_pt": str(best_pt),
}
if torch.cuda.is_available():
    results["peak_gpu_memory_gb"] = float(torch.cuda.max_memory_allocated() / 1024**3)
print(json.dumps(results, indent=2, ensure_ascii=False))


## 8. Results & Outputs

In [ ]:
# File lokal tetap tersimpan oleh Ultralytics, W&B menyimpan versi artifact tiap epoch.
run_dir.mkdir(parents=True, exist_ok=True)
results["best_epoch"] = wb_state["best_epoch"] if WANDB_ENABLED else None
(run_dir / "metrics.json").write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")
configuration = {
    "experiment": EXPERIMENT_ID, "dataset": DATA, "evaluation_split": EVAL_SPLIT,
    "model": MODEL_WEIGHTS, "teacher": TEACHER_CHECKPOINT if METHOD != "none" else None,
    "seed": SEED, "epochs": EPOCHS, "imgsz": IMGSZ, "batch": BATCH,
    "optimizer": OPTIMIZER, "lr0": LR, "device": DEVICE,
    "workers": WORKERS, "patience": PATIENCE, "amp": AMP,
    "weight_decay": WEIGHT_DECAY, "warmup_epochs": WARMUP_EPOCHS,
    "close_mosaic": CLOSE_MOSAIC,
    "kd_hyperparameters": KD_CONFIG,
    "wandb_enabled": WANDB_ENABLED, "wandb_project": WANDB_PROJECT,
}
(run_dir / "config.json").write_text(json.dumps(configuration, indent=2, ensure_ascii=False), encoding="utf-8")

if wb_run is not None:
    summary = {f"final/{key}": val for key, val in results.items()
               if isinstance(val, (int, float)) and not isinstance(val, bool) and val is not None}
    wb_run.log(summary)
    wb_run.summary.update(summary)
    wb_run.summary["best_epoch"] = wb_state["best_epoch"]
    wb_run.summary["checkpoint_collection"] = f"{EXPERIMENT_ID}-{wb_run.id}-checkpoints"
    wb_run.summary["uploaded_checkpoint_epochs"] = len(wb_state["uploaded_epochs"])

    # Ringkasan/config dan grafik sebagai Artifact terpisah, tanpa menggandakan checkpoint model.
    final_artifact = wandb.Artifact(
        name=f"{EXPERIMENT_ID}-{wb_run.id}-results", type="results",
        metadata={"experiment": EXPERIMENT_ID, "best_epoch": wb_state["best_epoch"]},
    )
    for filename in ("metrics.json", "config.json", "results.csv", "args.yaml"):
        candidate = run_dir / filename
        if candidate.is_file():
            final_artifact.add_file(str(candidate), name=filename)
    for folder_name, folder in (("training", run_dir),
                                ("evaluation", Path(getattr(metrics, "save_dir", run_dir)))):
        if folder.exists():
            for plot_path in sorted(folder.glob("*.png")):
                key = f"plots/{folder_name}/{plot_path.stem}"
                wb_run.log({key: wandb.Image(str(plot_path))})
                final_artifact.add_file(str(plot_path), name=f"{folder_name}/{plot_path.name}")
    wb_run.log_artifact(final_artifact, aliases=["final"]).wait()
    print("W&B:", wb_run.url)
    print("Checkpoint alias: epoch-001 ... epoch-%03d, latest, best" % len(wb_state["uploaded_epochs"]))
    wb_run.finish()

print("Hasil lokal:", run_dir)
print("  - weights/best.pt, weights/last.pt")
print("  - results.csv, args.yaml")
print("  - metrics.json, config.json")
